# 4-2회차 | 분류 ② KNN (K-Nearest Neighbors)

**핵심 질문**: "가까운 애들끼리 같은 라벨"은 어떻게 동작하는가?

**오늘의 목표**
1. **KNN 원리**: 거리 기반 다수결 분류
2. **K 값의 영향**: K가 작으면 과적합, K가 크면 과소적합
3. **스케일링의 중요성**: 단위가 다르면 거리 계산이 왜곡됨
4. **Decision Boundary 시각화**: K에 따라 경계가 어떻게 변하는지
5. **Pipeline + GridSearchCV**: 최적의 K 찾기

**예제**: Iris petal 2D (Decision Boundary), 스케일링 전후 비교

---

> **실행 전 준비물**
> - 데이터는 `load_iris()`로 자동 로드됨 — 별도 CSV 불필요
> - 슬라이드 PNG 8장 — 없으면 markdown 이미지가 깨짐. 코드 실행에는 지장 없음

---
## 로지스틱 회귀 vs KNN

4-1에서 로지스틱 회귀는 **확률**로 분류
이번에는 완전히 다른 방식 — **거리**로 분류하는 KNN을 배울거임

| 비교 | 로지스틱 회귀 | KNN |
|------|-------------|-----|
| 분류 기준 | 확률 (sigmoid) | 거리 (다수결) |
| 학습 방식 | 가중치(w, b) 학습 | 학습 없음 (데이터 저장) |
| 경계 모양 | 직선(선형) | 복잡한 곡선 |
| 스케일링 | 권장 | **필수** |

> 로지스틱 회귀: "점수가 높으면 양성"  
> KNN: "가까운 이웃이 양성이면 나도 양성"

---

### 잠깐 — KNN은 처음이 아님

앞 회차에서 펭귄 데이터로 이미 한 번 돌려봤음. 그때 나온 숫자:

> **펭귄 KNN Accuracy = 0.7801**

그때는 **모델을 블랙박스로 두고** 숫자만 받았음.
오늘은 그 안을 열어볼 거임 — 왜 0.78이었는지, 어떻게 하면 올라가는지.

> 오늘 끝날 때 이 숫자를 다시 꺼낼 거임. 기억해두셈.

---

## 오늘의 척추 질문

오늘 배울 것들은 따로따로 보이지만 **전부 하나의 질문**임.

> ### **"KNN에서 좋은 거리란 무엇인가?"**

| 오늘 다룰 것 | 사실은 이 질문임 |
|---|---|
| **스케일링** | 단위 때문에 거리가 **왜곡**되지는 않았나? |
| **피처 선택** | 쓸모없는 정보가 거리를 **오염**시키지는 않나? |
| **차원의 저주** | 좌표가 너무 많아서 거리 자체의 **의미가 약해**지지는 않나? |
| **K와 weights** | 찾은 이웃들의 의견을 **어떻게 모을** 것인가? |

> 오늘 끝날 때 이 질문에 답할 수 있으면 됨.
> 나머지는 전부 이 질문의 하위 항목임.

---
## Part 1. KNN 원리 — 거리 기반 다수결

### KNN이란?

1. 새로운 데이터 x가 들어오면
2. 학습 데이터에서 x와 **가까운 K개** 이웃을 찾고
3. 이웃 라벨의 **다수결**로 x의 라벨을 결정

### 예시 (K=3)

```
새 데이터 ★의 이웃 3개:
  ● 빨강 (거리=1.2)
  ● 빨강 (거리=1.5)
  ○ 파랑 (거리=2.0)

다수결: 빨강 2 > 파랑 1 → ★ = 빨강!
```

### 하이퍼파라미터: K (n_neighbors)

| K 값 | 특징 | 문제 |
|------|------|------|
| K 작음 (1~3) | 경계가 복잡 | 노이즈에 민감 → **과적합** |
| K 큼 (15~) | 경계가 단순 | 세부 패턴 놓침 → **과소적합** |
| K 적당 | 균형 | **교차검증**으로 찾기! |

> K를 찾는 건 결국 **교차검증(CV)**으로!

### 거리(Distance) 계산

KNN은 기본적으로 **유클리드 거리**(피타고라스)를 사용

$$d(A, B) = \sqrt{(x_1 - x_2)^2 + (y_1 - y_2)^2}$$

> 거리를 재기 때문에, **단위/범위가 다르면 큰 값이 거리를 지배**  
> 그래서 KNN에서는 **스케일링이 필수**!

---
## Part 2. Iris KNN 기본 사용법
- 참고) iris는 교육용 데이터셋이라 분리가 쉬움

In [1]:
import numpy as np, pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

iris = load_iris()
X, y = iris.data, iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

model = KNeighborsClassifier(
    n_neighbors=5,
    metric="minkowski",
    p=2  # 유클리드 거리
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(f"[Test Accuracy] {accuracy_score(y_test, y_pred):.3f}")
print()
print(classification_report(y_test, y_pred,
                            target_names=iris.target_names, digits=3))

[Test Accuracy] 1.000

              precision    recall  f1-score   support

      setosa      1.000     1.000     1.000        10
  versicolor      1.000     1.000     1.000        10
   virginica      1.000     1.000     1.000        10

    accuracy                          1.000        30
   macro avg      1.000     1.000     1.000        30
weighted avg      1.000     1.000     1.000        30



### Confusion Matrix — 어떤 클래스를 틀렸는지 확인

**3-2회차에서 배운 혼동행렬**을 다시 활용해보기.
3-2는 이진분류(7 vs 나머지)였고, 여기서는 **3개 클래스**임 — 표가 3×3이 됨.  
정확도만 보면 안 보이는 **"어떤 클래스끼리 헷갈리는지"**를 확인할 수 있음.

In [2]:
from sklearn.metrics import confusion_matrix
import plotly.figure_factory as ff

cm = confusion_matrix(y_test, y_pred)

fig = ff.create_annotated_heatmap(
    z=cm,
    x=list(iris.target_names),
    y=list(iris.target_names),
    colorscale="Blues", showscale=True
)
fig.update_layout(
    title="Confusion Matrix (KNN, K=5)",
    xaxis=dict(title="Predicted"),
    yaxis=dict(title="Actual"),
    template="plotly_dark"
)
fig.show()

> 지금은 4개 피처의 단위가 전부 cm라서 스케일링 없이도 잘 나옴.  
> **단위가 다른 피처가 섞이면 어떻게 되는지**를 바로 다음 파트에서 확인함.

---
## Part 3. 스케일링의 중요성

### 왜 KNN에서 스케일링이 필수인가?

KNN은 **거리**를 기반으로 판단함  
만약 두 특성의 **단위/범위**가 다르면?

| 특성 | 범위 | 거리 기여도 |
|------|------|:---------:|
| feature1 | 0 ~ 1 | 작음 |
| feature2 | 0 ~ 1000 | **엄청 큼** |

> feature2가 거리를 지배 → feature1은 무시됨  
> **스케일링으로 범위를 맞춰야 공정한 거리 계산 가능!**

In [3]:
from sklearn.preprocessing import StandardScaler

rng_scale = np.random.RandomState(42)
X_demo = np.c_[rng_scale.rand(200), rng_scale.rand(200) * 1000]
y_demo = (X_demo[:, 0] + X_demo[:, 1] / 1000 > 1.0).astype(int)

Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(
    X_demo, y_demo, test_size=0.3, stratify=y_demo, random_state=42
)

acc_raw = accuracy_score(
    yd_te,
    KNeighborsClassifier(5).fit(Xd_tr, yd_tr).predict(Xd_te)
)

sc = StandardScaler()
Xd_tr_s = sc.fit_transform(Xd_tr)
Xd_te_s = sc.transform(Xd_te)
acc_scaled = accuracy_score(
    yd_te,
    KNeighborsClassifier(5).fit(Xd_tr_s, yd_tr).predict(Xd_te_s)
)

pd.DataFrame({
    "설정": ["스케일링 없음", "스케일링(StandardScaler)"],
    "Accuracy": [acc_raw, acc_scaled]
})

,설정,Accuracy
0,스케일링 없음,0.650000
1,스케일링(StandardScaler),0.983333


> 스케일링 전후 Accuracy 차이가 **극적**임!  
> KNN에서 스케일링을 빼먹으면 성능이 폭락.

### 안전하게 스케일링하는 법: Pipeline

3회차에서 배운 Pipeline을 사용하면 **데이터 누수 없이** 스케일링 + 모델을 묶을 수 있음

```python
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier())
])
```

In [4]:
from sklearn.pipeline import Pipeline

pipe_knn = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=5))
])

knn_raw = KNeighborsClassifier(n_neighbors=5)
acc_raw = accuracy_score(y_test, knn_raw.fit(X_train, y_train).predict(X_test))

acc_pipe = accuracy_score(y_test, pipe_knn.fit(X_train,y_train).predict(X_test))

pd.DataFrame({
    "Model": ["KNN (스케일링 없음)", "Pipeline (Scaler + KNN)"],
    "Accuracy": [acc_raw, acc_pipe]
})

,Model,Accuracy
0,KNN (스케일링 없음),1.000000
1,Pipeline (Scaler + KNN),0.933333


### 잠깐 — 이 결과를 믿어도 됨?

방금 **한 번의 `train_test_split`** 으로 나온 숫자임.
3회차에서 배운 걸 떠올리셈: **한 번의 split으로 결론내지 않음.**

> **Prediction — 먼저 예상해보셈**: 교차검증으로 다시 재보면 이 하락이 그대로 나올 것 같음?
> 그리고 `random_state`를 바꾸면 어떻게 될 것 같음?

In [5]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

X_iris, y_iris = load_iris().data, load_iris().target

# (1) split을 20번 바꿔가며 비교
plain_scores, pipe_scores = [], []
for seed in range(20):
    Xa, Xb, ya, yb = train_test_split(
        X_iris, y_iris, test_size=0.2, stratify=y_iris, random_state=seed
    )
    plain_scores.append(
        KNeighborsClassifier(n_neighbors=5).fit(Xa, ya).score(Xb, yb)
    )
    pipe_scores.append(
        Pipeline([("scaler", StandardScaler()),
                  ("knn", KNeighborsClassifier(n_neighbors=5))]).fit(Xa, ya).score(Xb, yb)
    )

plain_scores, pipe_scores = np.array(plain_scores), np.array(pipe_scores)
print("[split 20번]")
print(f"  스케일링 없음이 더 높음 : {(plain_scores > pipe_scores).sum()} / 20")
print(f"  동점                  : {(plain_scores == pipe_scores).sum()} / 20")
print(f"  스케일링이 더 높음     : {(pipe_scores > plain_scores).sum()} / 20")
print()

# (2) 5-Fold 교차검증
cv_iris = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
s_plain = cross_val_score(KNeighborsClassifier(n_neighbors=5),
                          X_iris, y_iris, cv=cv_iris).mean()
s_pipe = cross_val_score(Pipeline([("scaler", StandardScaler()),
                                   ("knn", KNeighborsClassifier(n_neighbors=5))]),
                         X_iris, y_iris, cv=cv_iris).mean()
print("[5-Fold 교차검증]")
print(f"  스케일링 없음 : {s_plain:.4f}")
print(f"  스케일링      : {s_pipe:.4f}")


[split 20번]
  스케일링 없음이 더 높음 : 8 / 20
  동점                  : 11 / 20
  스케일링이 더 높음     : 1 / 20

[5-Fold 교차검증]
  스케일링 없음 : 0.9667
  스케일링      : 0.9733


### 결과 — "하락"은 한 번의 split이 만든 착시였음

- split을 20번 바꿔보면 **대부분 동점**이고, 스케일링이 이기는 경우도 있음
- **교차검증으로 재면 순서가 뒤집힘** — 스케일링 쪽이 오히려 높음

> 즉 `1.000 → 0.933`은 **`random_state=42`에서만 보이는 그림**임.

---

#### 그럼 아까 그 설명은 틀린 거임?

"판단 기준이 바뀌었다"는 설명 **자체는 맞음.** 스케일링은 이웃 관계를 바꿈.
틀린 건 **"성능이 떨어졌다"를 사실로 확정한 것**임.

> **오늘 가져갈 문장**:
> 스케일링은 정확도를 올리는 마법이 아님. **거리의 기준을 바꾸는 전처리**임.
> 그리고 **바뀐 결과가 좋은지 나쁜지는 한 번 돌려보고 정할 일이 아님.**

> **확인한 만큼만 말한다**: 이건 iris에서 확인한 것임.
> 단위가 크게 다른 데이터(바로 앞 파트의 0~1 vs 0~1000)에서는
> 스케일링 효과가 **훨씬 크고 안정적으로** 나타났음. 그건 앞 셀에서 봤음.

### Q. 스케일링을 하니까 오히려 성능이 떨어졌는데, 그럼 스케일링 안 하는 게 낫지 않나요?

**핵심 답변: 스케일링이 틀린 게 아니라, 모델의 판단 기준이 바뀐 거임**

위 Iris 결과를 보면 스케일링 없이 1.0, 스케일링 후 0.933이 나옴.
**단, 방금 확인했듯 이 하락 자체가 한 번의 split에서만 보이는 것임.**
그래도 "왜 이런 일이 생길 수 있는가"는 짚고 갈 필요가 있음.

---

**스케일링 전**: 범위가 큰 피처가 거리 계산을 **지배**.  
→ 모델이 사실상 **그 피처 하나만 보고** 분류한 것  
→ 우연히 그 피처가 분류에 유용했다면 성능이 높게 나올 수 있음

**스케일링 후**: 모든 피처를 **동일한 척도**로 비교함
→ 모델이 **모든 피처를 같은 무게로** 보고 분류
→ 판단 기준이 바뀌었기 때문에 성능 수치가 달라질 수 있음

> **주의 — 슬라이드의 "동등하게 기여"를 오해하면 안 됨**
> 스케일링이 맞춰주는 건 **숫자의 범위**지 **정보의 유용성**이 아님.
> `Age`와 `Fare`를 표준화했다고 해서 둘이 예측에 똑같이 쓸모 있어지는 건 아님.
> 단지 **"숫자가 크다는 이유만으로 목소리가 커지는 문제"**를 없애는 것임.

![스케일링 후: 모든 피처의 민주주의](스크린샷%202026-02-16%20오후%201.44.44.png)

---

### KNN은 어떤 모델인가?

![KNN의 정체성: 공정한 재판관](스크린샷%202026-02-16%20오후%201.45.13.png)

> **KNN = 공정한 거리 비교 모델**

- KNN은 특정 피처에 **가중치를 주는 모델이 아닙니다**
- 내가 입력한 피처들을 **동일하게 보고**, **동일한 기준(거리)**으로 비교하는 모델임
- 스케일링은 그 **"동일한 비교"를 가능하게 만들어주는 과정**

만약 피처 중요도를 학습하고 싶다면?  
→ 그건 **다른 모델**(로지스틱 회귀, 트리 모델 등)의 역할

### 그렇다면 KNN은 언제 쓰는 게 좋을까?

![그렇다면 KNN은 언제 써야 할까요?](스크린샷%202026-02-16%20오후%201.49.44.png)



---
## Part 4. K 값에 따른 성능 비교

### [4-2] Prediction Card 1 — K

> **K를 바꾸면 train 점수와 test 점수가 어떻게 움직임?**

**실행 전에** 예측부터 하셈.

| K | train 정확도 | test 정확도 |
|---|------------|------------|
| 1 | | |
| 5 | | |
| 15 | | |

같이 생각해볼 것:
- K=1일 때 **train** 정확도는 몇일 것 같음? 왜 그렇게 생각함?
- train과 test 중 **어느 쪽이 K에 더 민감**할 것 같음?

> 적었으면 실행하고, 예측과 얼마나 달랐는지 확인하셈.

In [6]:
import plotly.graph_objects as go
from sklearn.model_selection import cross_val_score, StratifiedKFold

k_list = [1, 3, 5, 7, 9, 11, 13, 15]
cv_k = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

train_list, cv_list = [], []

for k in k_list:
    pipe = Pipeline([("scaler", StandardScaler()),
                     ("knn", KNeighborsClassifier(n_neighbors=k))])

    # train 점수: 학습에 쓴 데이터를 그대로 예측
    pipe.fit(X_train, y_train)
    train_list.append(accuracy_score(y_train, pipe.predict(X_train)))

    # CV 점수: train 안에서만 5-Fold 교차검증 (test는 안 건드림)
    cv_list.append(
        cross_val_score(pipe, X_train, y_train, cv=cv_k, scoring="accuracy").mean()
    )

display(pd.DataFrame({
    "K": k_list,
    "train 정확도": np.round(train_list, 4),
    "CV 정확도": np.round(cv_list, 4),
    "격차 (train - CV)": np.round(np.array(train_list) - np.array(cv_list), 4),
}))

fig = go.Figure()
fig.add_trace(go.Scatter(x=k_list, y=train_list, mode="lines+markers",
                         name="train",
                         line=dict(color="grey", dash="dot")))
fig.add_trace(go.Scatter(x=k_list, y=cv_list, mode="lines+markers",
                         name="CV (5-Fold)",
                         line=dict(color="cyan")))
fig.update_layout(title="K 변화에 따른 train vs CV 정확도 (test는 사용하지 않음)",
                  xaxis_title="K (이웃 수)",
                  yaxis_title="Accuracy",
                  template="plotly_dark",
                  yaxis=dict(range=[0.88, 1.02]))
fig.show()


,K,train 정확도,CV 정확도,격차 (train - CV)
0,1,1.0000,0.9417,0.0583
1,3,0.9583,0.9667,-0.0083
2,5,0.9750,0.9583,0.0167
3,7,0.9750,0.9333,0.0417
4,9,0.9583,0.9583,0.0000
5,11,0.9583,0.9500,0.0083
6,13,0.9583,0.9500,0.0083
7,15,0.9667,0.9250,0.0417


---
## Part 5. Decision Boundary 시각화

K 값에 따라 **분류 경계(Decision Boundary)**가 어떻게 변하는지 직접 봅시다.  
Iris 데이터에서 petal length, petal width 2개 특성만 사용합니다.
- 이걸 가지고 과소적합, 과적합 확인해봅세

In [7]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
import plotly.graph_objects as go

iris = load_iris()
X_2d = iris.data[:, 2:4]
y_2d = iris.target

X_tr, X_te, y_tr, y_te = train_test_split(
    X_2d, y_2d, test_size=0.3, stratify=y_2d, random_state=42
)

pad = 0.5
xx, yy = np.meshgrid(
    np.linspace(X_2d[:, 0].min() - pad, X_2d[:, 0].max() + pad, 200),
    np.linspace(X_2d[:, 1].min() - pad, X_2d[:, 1].max() + pad, 200)
)
grid = np.c_[xx.ravel(), yy.ravel()]

k_list = [1, 3, 5, 7, 9, 15]
Z_map = {}
acc_map = {}

for k in k_list:
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=k))
    ])
    pipe.fit(X_tr, y_tr)
    Z_map[k] = pipe.predict(grid).reshape(xx.shape)
    acc_map[k] = pipe.score(X_te, y_te)

k0 = 5
fig = go.Figure()

fig.add_trace(
    go.Contour(
        x=xx[0], y=yy[:, 0], z=Z_map[k0],
        showscale=False, opacity=0.35,
        contours_coloring="heatmap",
        hoverinfo="skip", name="Decision Boundary"
    )
)

fig.add_trace(
    go.Scatter(
        x=X_tr[:, 0], y=X_tr[:, 1], mode="markers",
        marker=dict(symbol="circle", size=9,
                    color=y_tr, line=dict(width=1, color="white")),
        name="train"
    )
)

fig.add_trace(
    go.Scatter(
        x=X_te[:, 0], y=X_te[:, 1], mode="markers",
        marker=dict(symbol="triangle-up", size=11,
                    color=y_te, line=dict(width=1)),
        name="test"
    )
)

fig.update_layout(
    title=f"KNN Decision Boundary (K={k0}) — Test Acc: {acc_map[k0]:.3f}",
    xaxis_title="Petal length (cm)",
    yaxis_title="Petal width (cm)",
    template="plotly_dark",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig.update_layout(updatemenus=[
    dict(
        type="dropdown",
        x=1.0, xanchor="right", y=1.1, yanchor="top",
        buttons=[
            dict(
                label=f"K={k}",
                method="update",
                args=[
                    {"z": [Z_map[k], None, None]},
                    {"title": f"KNN Decision Boundary (K={k}) — Test Acc: {acc_map[k]:.3f}"}
                ]
            )
            for k in k_list
        ]
    )
])

fig.show()

### Decision Boundary 해석

드롭다운으로 K를 바꿔가며 **직접** 확인하셈. 아래 질문에 답할 수 있으면 됨.

- **K=1**과 **K=15**의 경계선 모양은 어떻게 다름?
- 경계가 울퉁불퉁한 것과 매끈한 것 중, 어느 쪽이 **과적합**임?
- 삼각형(test)이 잘못 분류된 지점이 K에 따라 어떻게 달라짐?

> 앞에서 본 train/test 점수 표와 이 그림을 **연결해서** 설명해보셈.
> "K가 작으면 과적합"이라는 문장이 이 그림의 어디에 보이는지 짚을 수 있어야 함.

### Q. 중요한 피처만 넣으면 성능이 더 좋아지지 않을까?

- 위 Decision Boundary를 보면, petal length와 petal width 2개만으로도 **거의 선형 분리**가 됨.
- 그렇다면 이 2개만 쓰는 게 4개 전부 쓰는 것보다 낫지 않을까?

직접 비교해 봅세.

In [8]:
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold

iris = load_iris()
X_all = iris.data
X_petal = iris.data[:, 2:4]
y = iris.target

pipe_all = Pipeline([("scaler", StandardScaler()),
                     ("knn", KNeighborsClassifier(n_neighbors=5))])
pipe_petal = Pipeline([("scaler", StandardScaler()),
                       ("knn", KNeighborsClassifier(n_neighbors=5))])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores_all = cross_val_score(pipe_all, X_all, y, cv=cv, scoring="accuracy")
scores_petal = cross_val_score(pipe_petal, X_petal, y, cv=cv, scoring="accuracy")

pd.DataFrame({
    "피처 구성": ["petal 2개 (length, width)", "전체 4개 피처"],
    "사용 피처 수": [2, 4],
    "CV 평균 Accuracy": [scores_petal.mean(), scores_all.mean()],
    "CV 표준편차": [scores_petal.std(), scores_all.std()]
})

,피처 구성,사용 피처 수,CV 평균 Accuracy,CV 표준편차
0,"petal 2개 (length, width)",2,0.960000,0.038873
1,전체 4개 피처,4,0.973333,0.024944


### 결과 해석: "중요한 피처만 남기면 더 좋아진다"는 착각

petal 2개만 썼을 때 4개 전부 쓴 것보다 평균이 **약간** 낮게 나옴.

> **잠깐 — 이 차이를 믿어도 됨?**
>
> | 피처 구성 | CV 평균 | CV 표준편차 |
> |---|---|---|
> | petal 2개 | 0.9600 | ±0.0389 |
> | 전체 4개 | 0.9733 | ±0.0249 |
>
> **차이는 0.0133인데 양쪽 표준편차보다 작음.**
> 즉 "4개가 확실히 더 좋다"고 단정할 근거는 **없음.**
> "이 CV에서는 평균이 조금 높게 관찰됐다" 까지만 말할 수 있음.

그래도 **왜 줄였는데 안 좋아졌는가**는 짚고 갈 가치가 있음. 왜?

> **"중요한 피처만 남기면 더 좋아질 것 같다"**  
> → 이건 **가중치 학습 모델의 사고 방식**임!

![모델마다 생각하는 방식이 다릅니다](스크린샷%202026-02-16%20오후%201.49.06.png)


| 사고 방식 | 해당 모델 | 무엇을 학습하나 |
|----------|----------|---------------|
| 피처별 중요도를 학습 | **로지스틱 회귀** | 계수 `w` (4-1에서 직접 봤음) |
| 피처별 중요도를 학습 | **트리 모델** | 분기 규칙 (`Fare < 20?`) → `feature_importances_` |
| 피처를 전부 거리식에 넣음 | **KNN** | **아무것도 학습 안 함** |

> 슬라이드는 로지스틱과 트리를 `w`로 묶었지만, **둘은 방식이 다름.**
> 로지스틱은 계수를 학습하고, 트리는 **분기 규칙**을 학습함.
> 공통점은 "중요도를 스스로 판단한다"는 것뿐임. 트리는 다음 시간에 확인함.

- KNN은 **가중치를 학습하지 않음**
- 그냥 모든 피처를 **다 더해서 거리를 계산**
- 그래서 피처를 줄이면 → **거리 계산에 쓸 단서 자체가 사라짐**

---

### 잠깐 — 앞에서 우리가 한 말이 서로 안 맞음

오늘 슬라이드 두 장을 나란히 놓아보셈.

| 언제 | 슬라이드가 한 말 |
|------|----------------|
| 스케일링 파트 | 무시되던 피처들이 계산에 포함되면서 **오히려 노이즈가 되었거나** 경계를 복잡하게 만듦 |
| 지금 | KNN에게 피처 삭제는 정제가 아니라 **'실명(blindness)'**에 가까움 |

- 앞에서는 피처가 늘어난 게 **노이즈**라고 했음
- 지금은 피처를 줄이는 게 **실명**이라고 함
- **둘 다 맞을 수는 없어 보이는데?**

> **오늘의 질문**: 어떤 피처를 빼는 게 '노이즈 제거'고, 어떤 피처를 빼는 게 '실명'임?
> 그 둘을 가르는 기준은 뭐임?

힌트가 될 만한 것:
- 방금 실험에서 뺀 건 sepal 2개였고, **성능이 떨어졌음**
- 만약 iris에 `측정한_날짜` 같은 컬럼이 있었다면? 그걸 빼면 성능이 올랐을까 떨어졌을까?

> **확인한 만큼만 말한다**: 지금 우리가 확인한 건 "이 데이터에서 sepal을 빼니 성능이 떨어졌다" 하나뿐임.
> "피처를 빼면 항상 나빠진다"는 아직 확인 안 한 말임.

---

### [4-2] Prediction Card 2 — 중요도

> **계수가 없는 KNN도 피처 중요도를 볼 수 있음?**

로지스틱은 4-1에서 `coef_`를 꺼내 봤음. KNN은 계수가 없음.
그럼 **KNN은 피처 중요도를 영영 못 보는 걸까?**

방법이 하나 있음 — **피처 하나를 무작위로 섞어버리고, 성능이 얼마나 떨어지는지 재는 것.**
많이 떨어지면 그 피처가 중요했던 거임. (**permutation importance**)

| 피처 | 섞었을 때 성능 하락 예상 |
|---|---|
| petal length / width | |
| sepal length / width | |

- 아까 "sepal을 빼면 정보 손실"이라고 했음. 그럼 sepal 중요도도 클까?

In [9]:
from sklearn.inspection import permutation_importance

X_pi, y_pi = load_iris().data, load_iris().target
feat_names = load_iris().feature_names

Xa, Xb, ya, yb = train_test_split(
    X_pi, y_pi, test_size=0.3, stratify=y_pi, random_state=42
)

pipe_pi = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=5))
]).fit(Xa, ya)

imp = permutation_importance(pipe_pi, Xb, yb, n_repeats=30,
                             random_state=42, scoring="accuracy")

display(
    pd.DataFrame({
        "피처": feat_names,
        "중요도(성능 하락폭)": imp.importances_mean.round(4),
        "± std": imp.importances_std.round(4),
    }).sort_values("중요도(성능 하락폭)", ascending=False).reset_index(drop=True)
)


,피처,중요도(성능 하락폭),± std
0,petal width (cm),0.2400,0.0442
1,petal length (cm),0.2030,0.0576
2,sepal length (cm),0.0067,0.0238
3,sepal width (cm),-0.0067,0.0288


### 결과 — 두 가지가 뒤집힘

**① "KNN은 피처 중요도를 못 본다"는 틀렸음**

계수가 없을 뿐, **섞어서 재는 방법(permutation importance)** 은 모델 종류를 안 가림.
KNN에도, 트리에도, 딥러닝에도 됨.

> 정확한 문장: KNN은 **학습된 계수가 없음.**
> 하지만 **사후에 중요도를 측정하는 건 가능함.**

**② sepal의 중요도가 사실상 0임**

`petal width`와 `petal length`는 중요도가 0.2 이상인데,
`sepal` 두 개는 **0에 붙어 있고 표준편차 안에서 음수도 나옴.**

> 그럼 아까 "sepal을 빼니 성능이 떨어졌다"는 건 뭐였음?
> **표준편차 안의 흔들림이었을 가능성이 큼.** (바로 위 표에서 확인했듯)

---

#### 그래서 결론을 다시 써야 함

| 처음 결론 | 확인 후 |
|---|---|
| "피처를 줄이면 정보가 줄어든다" | 이 데이터에서 sepal은 **거의 정보가 없었음** |
| "4개가 2개보다 좋다" | 차이가 std보다 작아 **단정 못 함** |

> **확인한 만큼만 말한다**: 그래도 "빼도 된다"는 결론은 아님.
> 중요도가 0에 가깝다는 것과 빼도 안전하다는 것은 다른 문제임.
> **다만 "KNN에서 피처를 빼면 무조건 나빠진다"는 말은 이 데이터가 지지하지 않음.**

---
## Part 6. GridSearchCV로 최적의 K 찾기

### 교차검증 + 그리드 탐색

K를 사람이 수동으로 하나씩 비교하지말고, **GridSearchCV**가 자동으로 최적의 K를 찾아줌

```python
param_grid = {
    "knn__n_neighbors": [1, 3, 5, 7, 9, 11],
    "knn__weights": ["uniform", "distance"],
}
```

- 각 조합에 대해 **5-Fold 교차검증** 수행
- 평균 정확도가 가장 높은 조합을 선택
- `refit=True`이면 최적 조합으로 **전체 데이터에 재학습**

---

### `weights`가 뭐임?

| 값 | 다수결 방식 |
|----|-----------|
| `uniform` | 이웃 K개가 **한 표씩** |
| `distance` | **가까운 이웃일수록 표가 무거움** (거리의 역수) |

> **⚠️ 슬라이드와 헷갈리지 말 것 — "가중치"가 두 종류임**
>
> | 종류 | 무엇에 대한 가중치 | KNN이 학습함? |
> |---|---|:---:|
> | **피처 가중치** | "Age가 Fare보다 중요한가?" | **안 함** ← 슬라이드가 말한 그거 |
> | **이웃 투표 가중치** (`weights`) | "가까운 이웃 표를 더 세게 칠까?" | 우리가 **직접 정함** |
>
> 슬라이드의 "KNN은 가중치가 없다"는 **피처 가중치** 얘기임.
> `weights=` 파라미터는 **이웃 투표** 얘기라 서로 다른 것임.
> KNN은 여전히 "어떤 피처가 중요한지"는 **스스로 배우지 않음.**

우리는 앞에서 이렇게 배웠음:

> **K가 작으면 과적합, K가 크면 과소적합**

그런데 `distance`를 넣고 돌리면 이 문장이 흔들림.
다음 셀에서 **train 점수**를 같이 보면 뭔가 이상한 게 보일 거임.

> **Prediction — 먼저 예상해보셈**: `distance`로 K=11을 쓰면 train 정확도는 몇일 것 같음?

In [10]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier())
])

param_grid = {
    "knn__n_neighbors": [1, 3, 5, 7, 9, 11],
    "knn__weights": ["uniform", "distance"],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

gs = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    scoring="accuracy",
    cv=cv,
    n_jobs=-1,
    refit=True,
    return_train_score=True
)

iris = load_iris()

# 최종 시험지(test)를 먼저 떼어놓고, GridSearch는 남은 데이터에서만 함
X_dev, X_final_test, y_dev, y_final_test = train_test_split(
    iris.data, iris.target, test_size=0.2, stratify=iris.target, random_state=42
)

gs.fit(X_dev, y_dev)

print(f"탐색 범위 (dev) : {len(X_dev)}개")
print(f"봉인 (test) : {len(X_final_test)}개  ← 맨 마지막에 딱 한 번")
print()
print(f"CV 최고 조합    : {gs.best_params_}")
print(f"CV 최고 Accuracy: {gs.best_score_:.4f}")

탐색 범위 (dev) : 120개
봉인 (test) : 30개  ← 맨 마지막에 딱 한 번

CV 최고 조합    : {'knn__n_neighbors': 3, 'knn__weights': 'uniform'}
CV 최고 Accuracy: 0.9667


In [11]:
import pandas as pd

cv_df = pd.DataFrame(gs.cv_results_)
cv_table = cv_df[[
    "param_knn__n_neighbors", "param_knn__weights",
    "mean_train_score", "std_train_score",
    "mean_test_score", "std_test_score", "rank_test_score"
]].sort_values(["param_knn__weights", "param_knn__n_neighbors"])

cv_table.rename(columns={
    "param_knn__n_neighbors": "K",
    "param_knn__weights": "weights",
    "mean_train_score": "train_acc",
    "mean_test_score": "cv_mean_acc",
    "std_test_score": "cv_std_acc"
}, inplace=True)

display(cv_table)


,K,weights,train_acc,std_train_score,cv_mean_acc,cv_std_acc,rank_test_score
1,1,distance,1.000000,0.000000,0.941667,0.020412,9
3,3,distance,1.000000,0.000000,0.966667,0.016667,1
5,5,distance,1.000000,0.000000,0.958333,0.026352,4
7,7,distance,1.000000,0.000000,0.941667,0.020412,9
9,9,distance,1.000000,0.000000,0.958333,0.026352,4
11,11,distance,1.000000,0.000000,0.966667,0.016667,1
0,1,uniform,1.000000,0.000000,0.941667,0.020412,9
2,3,uniform,0.968750,0.009317,0.966667,0.016667,1
4,5,uniform,0.968750,0.006588,0.958333,0.026352,4
6,7,uniform,0.966667,0.010206,0.933333,0.020412,12


In [12]:
import plotly.graph_objects as go

colors = {"uniform": "cyan", "distance": "magenta"}
fig = go.Figure()

for w in ["uniform", "distance"]:
    sub = cv_table[cv_table["weights"] == w].sort_values("K")

    fig.add_trace(go.Scatter(
        x=list(sub["K"]) + list(sub["K"])[::-1],
        y=list(sub["cv_mean_acc"] + sub["cv_std_acc"]) +
          list(sub["cv_mean_acc"] - sub["cv_std_acc"])[::-1],
        fill="toself", opacity=0.15,
        line=dict(color=colors[w], width=0),
        hoverinfo="skip", showlegend=False
    ))

    fig.add_trace(go.Scatter(
        x=sub["K"], y=sub["cv_mean_acc"],
        mode="lines+markers", name=f"CV ({w})",
        line=dict(color=colors[w])
    ))

    fig.add_trace(go.Scatter(
        x=sub["K"], y=sub["train_acc"],
        mode="lines+markers", name=f"train ({w})",
        line=dict(color=colors[w], dash="dot")
    ))

fig.update_layout(
    title="KNN GridSearchCV 결과 — weights별 train vs CV (mean ± std)",
    xaxis_title="K (이웃 수)",
    yaxis_title="Accuracy",
    template="plotly_dark"
)
fig.show()


### GridSearchCV 결과 해석

- `cv_mean_acc`: 5-Fold 교차검증 평균 정확도
- `cv_std_acc`: 분산 (작을수록 안정적)
- `rank_test_score`: 순위 (1이 최고)

> **K=1**(과적합)은 훈련 점수는 높지만 CV 점수는 낮을 수 있음

---

### 잠깐 — `best_params_`가 진짜 "유일한 승자"임?

위 표에서 **`rank_test_score = 1`이 몇 줄인지** 세어보셈.

한 줄이 아니라면, `best_params_`가 고른 조합은
성능이 더 좋아서가 아니라 **먼저 나온 순서대로 잘랐기 때문**임 (tie-breaking).

> **그럼 뭘 골라야 함?**
> 성능이 같으면 **더 단순한 쪽**을 고르는 게 기본임. `uniform`이 더 단순함.
> 또는 **반복 교차검증**으로 다시 재서 차이가 있는지 확인함.

> **하면 안 되는 말**: "최적 조합은 K=5, uniform입니다"
> **해야 하는 말**: "현재 CV에서는 K=5의 두 weights가 공동 1위입니다"

---

### `distance`의 train 곡선을 보셈

점선(train)이 `distance`에서는 **K와 상관없이 1.000에 붙어 있음**.

왜 그럼?
- train 데이터를 예측할 때는 **자기 자신이 이웃에 포함**됨
- 자기 자신까지의 거리는 0 → 거리의 역수 가중치가 **무한대에 가까움**
- 그래서 나머지 이웃이 뭐라 하든 **자기 라벨이 항상 이김**

> **그럼 "K를 키우면 과적합이 줄어든다"는 말은 틀린 거임?**
> 아님. 조건이 빠진 거임 — 그 문장은 `uniform`일 때의 이야기였음.
> 실선(CV)을 보면 `distance`도 K에 따라 움직임. **과적합 여부는 train 점수가 아니라 train-CV 격차로 봐야 함.**

> **확인한 만큼만 말한다**: 위 설명은 iris + `weights="distance"`에서 관찰한 것임.
> 다른 데이터에서도 똑같은지는 직접 돌려봐야 앎.


In [13]:
ties = cv_table[cv_table["rank_test_score"] == 1]

print(f"공동 1위 조합 수: {len(ties)}개")
display(ties[["K", "weights", "cv_mean_acc", "cv_std_acc"]])

print()
print(f"best_params_ 가 고른 것 : {gs.best_params_}")
print(f"실제로 1위인 조합       : {len(ties)}개 (소수점까지 동일)")


공동 1위 조합 수: 3개


,K,weights,cv_mean_acc,cv_std_acc
3,3,distance,0.966667,0.016667
11,11,distance,0.966667,0.016667
2,3,uniform,0.966667,0.016667



best_params_ 가 고른 것 : {'knn__n_neighbors': 3, 'knn__weights': 'uniform'}
실제로 1위인 조합       : 3개 (소수점까지 동일)


### 결과 — 승자가 여러 명임

`best_params_`는 **하나만 돌려주지만**, 실제로는 공동 1위가 여럿임.
`GridSearchCV`는 동점일 때 **먼저 계산된 것**을 고름. 성능 우열이 아님.

> **그럼 뭘 골라야 함?**
>
> | 기준 | 선택 |
> |---|---|
> | **더 단순한 쪽** | `uniform`이 `distance`보다 단순 |
> | **K가 큰 쪽** | 노이즈에 덜 민감 |
> | **다시 재보기** | 반복 교차검증(`RepeatedStratifiedKFold`)으로 차이가 진짜인지 확인 |

> **하면 안 되는 말**: "최적 조합은 K=◯◯, ◯◯입니다"
> **해야 하는 말**: "현재 CV에서는 이 조합들이 **공동 1위**입니다"

> **확인한 만큼만 말한다**: 동점 개수와 어느 조합이 1위인지는
> **split과 CV seed를 바꾸면 달라짐.** 그래서 하나를 "정답"이라 부르면 안 됨.

---
## Part 7. 차원의 저주 — KNN은 왜 고차원에서 약해질까?

### Q. 4개 피처에서는 2개보다 성능이 좋았는데, 피처를 계속 늘리면 어떻게 될까?

![피처가 많을수록 정보가 많으니 무조건 좋을까요?](스크린샷%202026-02-16%20오후%201.50.08.png)


아까 피처 4개 > 피처 2개였으니, 계속 늘리면 더 좋아지지 않을까?

> **핵심 답변: 적당한 차원 증가 = 정보 증가, 과도한 차원 증가 = 공간 붕괴**

### 2차원에서의 "가까움"

2차원에서는 직관이 잘 작동함:
- 가까운 점은 **확실히 가깝고**
- 먼 점은 **확실히 멀다**

```
  ●  ← 가까움 (거리 = 1.2)
★
        ○  ← 멈 (거리 = 8.5)
```

### 고차원에서의 "가까움" 붕괴

![차원의 저주: 공간이 붕괴된다](스크린샷%202026-02-16%20오후%201.50.22.png)


하지만 차원이 늘어나면(피처 수가 많아지면):
- 모든 점들이 **비슷한 거리로 퍼짐**
- 가장 가까운 점과 가장 먼 점의 **거리 차이가 줄어듦**

![모든 것이 멀어지는 세계](스크린샷%202026-02-16%20오후%201.50.37.png)

$$\text{차원이 증가할수록:} \quad \frac{\text{가장 가까운 거리}}{\text{가장 먼 거리}} \to 1$$

> **가까운 것도 멀고, 먼 것도 멀다 → 결국 다 멀다!**  
> 이것을 **차원의 저주(Curse of Dimensionality)**라고 함.

### 왜 KNN에 치명적인가?

| KNN의 핵심 | 고차원에서 벌어지는 일 |
|-----------|-------------------|
| 거리가 가까운 이웃의 **다수결**로 판단 | 모든 점이 거의 같은 거리 |
| **이웃** 개념이 핵심 | 이웃 개념이 **무의미**해짐 |
| 지역적(local) 판단 모델 | **노이즈에 매우 민감**해짐 |

→ 그래서 KNN은 고차원에서 **급격하게 성능이 떨어짐**

In [14]:
import numpy as np
from scipy.spatial.distance import cdist
import plotly.graph_objects as go

rng_cd = np.random.RandomState(42)
dims = [2, 10, 50, 200, 500]
ratios = []

for d in dims:
    X = rng_cd.rand(200, d)
    D = cdist(X, X)

    np.fill_diagonal(D, np.inf)
    min_dist = D.min(axis=1)

    np.fill_diagonal(D, 0)
    max_dist = D.max(axis=1)

    ratios.append((min_dist / max_dist).mean())

fig = go.Figure()
fig.add_trace(go.Scatter(x=dims, y=ratios, mode="lines+markers"))
fig.update_layout(
    title="차원이 커질수록 가까운 거리 / 먼 거리 → 1",
    xaxis_title="차원 수",
    yaxis_title="min / max 거리 비율",
    template="plotly_dark"
)
fig.show()


### [4-2] Prediction Card 3 — 노이즈

> **정답과 무관한 컬럼을 붙이면 KNN은 알아서 무시함?**

위 그래프는 **거리 비율**이 무너지는 걸 보여줬음.
그런데 우리가 진짜 알고 싶은 건 **성능**임.

iris(4차원, CV Accuracy 0.9733)에 **완전히 무작위인 숫자 컬럼**을 붙여볼 거임.
그 컬럼들은 붓꽃 품종과 **아무 상관이 없음.**

| 노이즈 피처 추가 | CV Accuracy 예상 |
|----------------|----------------|
| 2개 | |
| 20개 | |
| 200개 | |

- 쓸모없는 컬럼이니까 모델이 **알아서 무시**할 것 같음?
- 아니면 성능이 떨어질 것 같음? 떨어진다면 얼마나?
- 랜덤 추측은 정확도 **0.333**임 (품종 3개). 거기까지 내려갈 것 같음?

> 적었으면 실행하셈.

In [15]:
import numpy as np
import pandas as pd
from sklearn.model_selection import cross_val_score, StratifiedKFold

iris = load_iris()
X_base, y_base = iris.data, iris.target
n_list = [0, 2, 5, 10, 20, 50, 100, 200]

rows = []
for seed in range(5):
    rng_noise = np.random.RandomState(seed)
    # 노이즈 200개를 한 번만 만들고, 앞에서부터 잘라 씀 (누적 실험)
    noise_pool = rng_noise.rand(len(X_base), max(n_list))
    cv_noise = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)

    for n_noise in n_list:
        X_aug = np.hstack([X_base, noise_pool[:, :n_noise]]) if n_noise else X_base
        pipe_noise = Pipeline([
            ("scaler", StandardScaler()),
            ("knn", KNeighborsClassifier(n_neighbors=5))
        ])
        score = cross_val_score(pipe_noise, X_aug, y_base,
                                cv=cv_noise, scoring="accuracy").mean()
        rows.append([seed, X_aug.shape[1], n_noise, score])

raw_noise = pd.DataFrame(rows, columns=["seed", "총 차원", "노이즈 수", "acc"])
noise_df = (raw_noise.groupby(["노이즈 수", "총 차원"])["acc"]
            .agg(평균="mean", 표준편차="std").round(4).reset_index())
display(noise_df)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=noise_df["총 차원"], y=noise_df["평균"],
    error_y=dict(type="data", array=noise_df["표준편차"], visible=True),
    mode="lines+markers", name="KNN CV Accuracy (seed 5개 평균)",
    line=dict(color="magenta")
))
fig.add_hline(y=1/3, line_dash="dash", line_color="grey",
              annotation_text="랜덤 추측 (1/3)")
fig.update_layout(
    title="의미 없는 피처를 늘릴수록 KNN 성능이 무너짐 (seed 5개, 오차막대 = ±1 std)",
    xaxis_title="총 차원 수 (로그 스케일)", xaxis_type="log",
    yaxis_title="CV Accuracy", yaxis=dict(range=[0, 1.05]),
    template="plotly_dark"
)
fig.show()


,노이즈 수,총 차원,평균,표준편차
0,0,4,0.9520,0.0073
1,2,6,0.8933,0.0200
2,5,9,0.8360,0.0252
3,10,14,0.8053,0.0242
4,20,24,0.7507,0.0361
5,50,54,0.6867,0.0641
6,100,104,0.6027,0.0289
7,200,204,0.5253,0.0292


### 결과 — 그리고 아까 던진 질문의 답

노이즈 컬럼은 품종과 **아무 상관이 없는데도** 성능이 무너짐.
KNN은 가중치를 학습하지 않으니까 **쓸모없는 컬럼도 거리 계산에 그대로 합산**됨.
컬럼이 많아질수록 진짜 신호(petal)가 노이즈 합계에 파묻힘.

---

#### Part 5에서 던진 질문으로 돌아가셈

> **어떤 피처를 빼는 게 '노이즈 제거'고, 어떤 피처를 빼는 게 '실명'임?**

이제 같은 노트북 안에 두 사례가 다 있음.

| 무엇을 뺐나 | 성능 변화 | 표준편차 대비 | 뭐라고 불러야 함? |
|------------|----------|---|-----------------|
| sepal 2개 (permutation 중요도 ≈ 0) | 0.9733 → 0.9600, 아주 조금 하락 | **std 안** — 불확실 | 판단 보류 |
| 노이즈 200개 | 위 표에서 **0.95 → 0.53** 급락이 역전됨 | **std 밖** — 명백 | **노이즈 제거** |

> **기준은 "피처 개수"가 아니라 "그 피처가 정답과 관계가 있는가"임.**
> 그리고 **관계가 있는지 없는지는 오차막대까지 보고 판단해야 함.**
> 위 그래프의 오차막대가 겹치는 구간에서는 "떨어졌다"고 말하면 안 됨.

그래서 슬라이드 두 장이 둘 다 맞았음:
- 스케일링 슬라이드의 "노이즈가 되었거나" → **무관한 피처**를 두고 한 말
- "피처 삭제 = 실명" 슬라이드 → **유의미한 피처**를 두고 한 말

**빠져 있던 건 "어느 쪽인지 어떻게 아느냐"였고, 답은 "돌려봐야 안다"임.**

> **확인한 만큼만 말한다**: 위 표는 iris에서 직접 돌린 결과임.
> "무관한 피처는 빼는 게 낫다"는 일반 원칙으로는 그럴듯하지만,
> 어떤 피처가 무관한지는 **데이터마다 확인해야 아는 것**임.

---

#### 앞 회차 펭귄 0.7801로 돌아가보셈

도입부에서 꺼낸 그 숫자를 기억하셈.

- 그때 스케일링을 했었음? 안 했었음?
- 그때 쓴 피처 중에 **정답과 무관한 것**이 섞여 있었을까?
- K는 몇이었고, 교차검증으로 고른 값이었음?

> 오늘 배운 걸로 그 0.7801을 **다시 설명할 수 있으면** 이번 회차는 성공임.

### 차원의 저주 정리

| 차원 증가 정도 | 효과 | 이 노트북에서 확인한 값 |
|:----------:|:----:|:----|
| **적당히** — 의미 있는 피처 2→4 | 차이가 **std 안** | 0.9600 vs 0.9733 — 단정 못 함 |
| **과도하게** — 무의미한 피처 +50 | 성능 **하락** | 0.9520 → **0.6867** |
| **극단적으로** — 무의미한 피처 +200 | 성능 **붕괴** | 0.9520 → **0.5253** |

> 위 두 줄은 **오차막대 밖**이라 확실하고, 첫 줄은 **오차막대 안**이라 불확실함.
> 같은 표 안에서도 **믿을 수 있는 줄과 없는 줄이 섞여 있음.**

> 주의: 차원이 늘어서 무너진 게 아니라, **정답과 무관한 차원**이 늘어서 무너진 거임.

---

### 한 가지 더 — `min / max → 1`은 보편 법칙이 아님

슬라이드의 수식은 **직관을 위한 그림**이지 모든 데이터에 성립하는 법칙이 아님.
같은 실험을 데이터 분포만 바꿔서 돌리면 이렇게 나옴.

| 데이터 분포 | d=2 | d=10 | d=50 | d=200 |
|---|---|---|---|---|
| 균등난수 (앞 셀과 동일) | 0.036 | 0.332 | 0.644 | **0.807** |
| 정규분포 | 0.036 | 0.319 | 0.602 | 0.784 |
| **군집이 뚜렷한 데이터** | 0.014 | 0.059 | 0.089 | **0.103** |

- 균등난수조차 200차원에서 **0.807**이지 1이 아님
- **구조(군집)가 있으면 200차원에서도 0.103** — 거의 안 무너짐

> **정리**: 차원의 저주는 **"데이터에 구조가 없을 때"** 심해짐.
> 우리가 붙인 노이즈 피처가 바로 "구조 없는 차원"이었음.
> 그래서 성능이 무너진 거지, 차원 숫자 자체가 죄는 아님.

> **확인한 만큼만 말한다**: 위 표는 인공 데이터로 돌린 결과임.
> 실제 데이터가 어느 쪽에 가까운지는 **돌려봐야 앎.**

> **적당한 차원 증가 = 정보 증가**  
> **과도한 차원 증가 = 거리 구별력 붕괴 (차원의 저주)**

### 고차원 데이터를 다루려면?

KNN은 고차원에서 구조적으로 약하기 때문에, 고차원 데이터에는 다른 전략이 필요함:
- **차원 축소** (PCA 등)
- **Feature Selection** (유용한 피처만 선별)
- **다른 모델** 사용 (로지스틱 회귀, 트리 모델, SVM 등)

> 다음 시간에 배울 **트리 모델(Decision Tree)**은 왜 고차원에서 강한지도 비교해 보겠음!

![요약: KNN 마스터를 위한 3가지 핵심](스크린샷%202026-02-16%20오후%201.51.23.png)

---

## Part 8. 마지막 — 수능 보러 가기

4-1에서 한 것과 똑같고, **3-2회차에서 threshold를 고를 때 한 것과도 똑같음.**

> 배울 때는 train만 봄. 선택할 때는 validation을 봄.
> test는 모든 선택이 끝난 뒤 마지막에 한 번 봄.

지금까지 우리가 고른 것들:

| 무엇을 골랐나 | 어디서 |
|---|---|
| 스케일링 여부 | CV |
| 피처 구성 | CV |
| K와 weights | CV (GridSearchCV) |

**전부 dev 안에서 골랐음.** `X_final_test`은 아직 한 번도 안 썼음.

### [4-2] Prediction Card 4 — 최종 시험

바로 위에서 나온 CV 최고 점수를 확인하고, test 점수를 예상해보셈.

- 더 높을까, 낮을까?
- **30개짜리 test**임. 한 명 틀리면 정확도가 얼마나 움직임?

In [16]:
best_model = gs.best_estimator_
final_test_acc = best_model.score(X_final_test, y_final_test)

print(f"CV 최고 (dev 안) : {gs.best_score_:.4f}")
print(f"test (수능)   : {final_test_acc:.4f}   (n={len(y_final_test)})")
print()
print(f"test에서 1명 틀릴 때 정확도 변화폭: {1/len(y_final_test):.4f}")


CV 최고 (dev 안) : 0.9667
test (수능)   : 0.9333   (n=30)

test에서 1명 틀릴 때 정확도 변화폭: 0.0333


### 해석 — 숫자보다 중요한 것

test이 30개뿐임. **한 명만 틀려도 정확도가 0.0333씩 움직임.**
그러니 `0.9667`과 `1.0000`의 차이는 **한 명 차이**임.

> **그래서 하면 안 되는 말**: "test에서 100% 나왔으니 완벽한 모델입니다"
> **해야 하는 말**: "test 30개 기준이라 이 숫자 하나로는 확정 못 합니다"

---

#### 4-1과 4-2가 같은 말을 하고 있음

| | 4-1 로지스틱 | 4-2 KNN |
|---|---|---|
| valid/CV에서 고른 것 | 임곗값, 피처 | 스케일링, 피처, K, weights |
| test 역할 | 딱 한 번 평가 | 딱 한 번 평가 |
| 배운 것 | **고르는 데 쓴 점수는 성능이 아님** | 동일 |

> 모델이 확률이든 거리든, **평가 규칙은 똑같음.**
> 이게 3회차부터 이어져 온 하나의 원칙임.

---
## 오늘의 정리

| 개념 | 핵심 | 코드 |
|------|------|------|
| KNN | 거리 기반, 이웃 K개 다수결 | `KNeighborsClassifier(n_neighbors=K)` |
| 스케일링 | KNN에서 **필수** (거리 왜곡 방지) | `StandardScaler()` |
| Pipeline | 스케일링 + 모델을 안전하게 묶기 | `Pipeline([("scaler", ...), ("knn", ...)])` |
| K 탐색 | 교차검증으로 최적 K 찾기 | `GridSearchCV(pipe, param_grid, cv=cv)` |
| Decision Boundary | K에 따라 경계 복잡도 변화 | K 작음=과적합, K 큼=과소적합 |
| 차원의 저주 | 고차원에서 거리 구별력 붕괴 | 적당한 차원↑=정보↑, 과도한 차원↑=성능↓ |
| weights | `distance`는 train 점수를 1.0으로 밀어버림 | 과적합은 train-CV **격차**로 판단 |

### 오늘의 핵심 Q&A 요약(지난 기수)

| 질문 | 핵심 답변 |
|------|----------|
| 스케일링하면 성능이 떨어지는데? | 모델의 판단 기준이 바뀐 것. 스케일링이 "공정한 비교"를 만드는 것 |
| KNN은 언제 쓰는 게 좋은가? | 모든 피처를 동일한 척도로 비교하여 유사성 기반 분류할 때 |
| 중요한 피처만 남기면 더 좋지 않나? | KNN은 가중치 학습 모델이 아님. **유의미한** 피처를 빼면 단서가 사라짐 — 다만 '유의미한지'는 확인해봐야 아는 것 |
| 왜 고차원에서 약해지나? | 차원의 저주: 모든 점이 비슷한 거리 → 이웃 개념이 무의미 |

### 로지스틱 회귀 vs KNN 최종 비교

| 비교 | 로지스틱 회귀 | KNN |
|------|-------------|-----|
| 분류 기준 | **확률** (sigmoid) | **거리** (다수결) |
| 학습 | 가중치 학습 (빠름) | 학습 없음 (예측 시 계산) |
| 피처 중요도 | **학습함** (계수 = 중요도) | **학습 안 함** (모두 동일하게 취급) |
| 경계 | 직선 (선형) | 복잡한 곡선 |
| 스케일링 | 권장 | **필수** |
| 고차원 | 상대적으로 강함 | **차원의 저주에 취약** |
| 장점 | 확률 출력, 해석 쉬움 | 구현 쉬움, 비선형 경계 |
| 단점 | 비선형 패턴 약함 | 데이터 많으면 느림, 고차원 취약 |
| 핵심 하이퍼파라미터 | C (규제 강도) | K (이웃 수) |

---

## 척추 질문으로 돌아가기

> ### **"KNN에서 좋은 거리란 무엇인가?"**

오늘 확인한 답:

| 질문 | 오늘 확인한 것 |
|---|---|
| 단위 때문에 **왜곡**되지 않았나? | 0~1 vs 0~1000 → 0.650, 스케일링 후 0.983 |
| 쓸모없는 정보가 **오염**시키지 않나? | 노이즈 200개 → **0.95에서 0.53으로 붕괴** |
| 어떤 피처가 실제로 쓰이나? | permutation 중요도: petal ≫ sepal(≈0) |
| 좌표가 많아 **의미가 약해**지지 않나? | 구조 없는 데이터일수록 심함 (0.807 vs 0.103) |
| 이웃 의견을 **어떻게 모을까**? | K와 `weights` — CV로 고름 (단, **공동 1위 여럿**) |
| 그래서 성능은? | valid/CV가 아니라 **test에서 딱 한 번** 확인 |

> **KNN을 잘 쓴다는 건 결국 "좋은 이웃을 찾을 수 있는 공간"을 만드는 일임.**
> 모델을 튜닝하는 게 아니라 **거리를 설계하는 것**임.

---

> **기억할 문장**:  
> 로지스틱 = "확률로 분류", KNN = "가까운 애들끼리 다수결로 분류"  
> KNN은 **공정한 거리 비교 모델** — 특정 피처에 힘을 실어주지 않는다!  
> 둘 다 **스케일링 + Pipeline + 교차검증**이 기본!

---
## 다음 시간 예고

- **결정 트리 (Decision Tree)**: 질문을 반복해서 분류하는 규칙 기반 모델
- **앙상블 (Ensemble)**: 여러 모델을 합쳐서 더 좋은 성능 만들기
- **Bias-Variance Tradeoff**: 과적합/과소적합의 근본 원리

> 오늘 확률(로지스틱)과 거리(KNN) 기반 분류를 배웠으니,  
> 다음엔 **규칙 기반** 분류인 결정 트리를 배울거임!
>
> 트리 모델은 KNN과 달리 **분기 규칙**을 학습하고,
> 그 덕에 **무관한 피처를 아예 안 쓰고 넘어갈 수 있음.**
>
> 단, "트리는 고차원에서 항상 강하다"는 **아님.**
> 무관한 피처가 아주 많으면 트리도 잘못된 분기를 고를 수 있음.
> 어느 쪽이 어떤 조건에서 강한지 다음 시간에 **직접 돌려서** 비교할 거임!
>
> 숙제 파일(`4회차_숙제.ipynb`)을 확인하고 생각해오셈!